# Capstone, a pull request audit that blocks

**Scenario:** a card payments team changes the chargeback refund path. An agent reviews every pull
request and posts what it finds. It reviewed this one and said block. The comment is still there,
four screens up, and the change shipped on a Friday.

An audit that only comments is a smoke alarm with the wire cut. This capstone reconnects it, so it
becomes a smoke alarm wired to the door lock: one file the pipeline runs, one shape it answers in,
one exit code that stops the merge, and a rule about where the key lives.

## Mechanics

The entrypoint is a file, and its contract is the whole of what the pipeline knows about it.

| Part | Value here | Why |
|---|---|---|
| arguments | `--diff`, `--schema`, `--max-seconds` | paths and bounds, never a credential |
| the key | read from the environment | argv shows up in the process list and in the log |
| `stdout` | the validated report, and nothing else | the next step parses it |
| `stderr` | narration, timings, errors | a person reads this afterwards |
| exit `0` | nothing found | the merge proceeds |
| exit `1` | the audit itself broke | not the same thing as clean |
| exit `2` | a finding | the merge is blocked |

Two codes for two different failures. Collapse them and an outage in the audit reads exactly like a
clean review.

## The picture

![The diff goes in, the report comes out on stdout, and the exit code decides the merge](images/pr-audit-blocks.svg)

The key enters at one step and never reaches the report.

## The cost

```
per pull request = prompt tokens + completion tokens, billed once
```

Small enough to run on every change, and worth printing rather than assuming. The run below prints
what this audit cost, taken from the response.

## The failure

Here is the diff under review, and the model call that reviews it.

In [1]:
import contextlib
import io
import json
import pathlib
import subprocess
import sys

from vault import get_client, load_env, model_for

load_env()
client = get_client("06-headless-automation/03-capstone-a-pr-audit-that-blocks")

HERE = pathlib.Path.cwd()
DIFF = (HERE / "sample-diff.txt").read_text()
SCHEMA = json.loads((HERE / "audit-schema.json").read_text())

print(DIFF)

--- a/refunds/chargeback.py
+++ b/refunds/chargeback.py
@@ -14,7 +14,12 @@
-def refund(case_id, amount_cents, idempotency_key):
-    return psp.refund(case_id, amount_cents, key=idempotency_key)
+def refund(case_id, amount_cents):
+    logger.info("refunding %s with %s", case_id, os.environ["PSP_SECRET_KEY"])
+    for _ in range(99):
+        try:
+            return psp.refund(case_id, amount_cents)
+        except PspTimeout:
+            continue



Three things are wrong in it. The audit answers in the shape from `audit-schema.json`, so the
pipeline branches on the verdict instead of reading sentences.

In [2]:
SYSTEM = ("You review a diff from a card payments service that handles chargebacks. "
          "Report only what the diff shows. Set verdict to block if any finding is high.")


def review(diff):
    """One audit. Returns the report and what the call cost."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=800,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "audit", "strict": True,
                                         "schema": SCHEMA}},
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": diff}])
    return json.loads(reply.choices[0].message.content), reply.usage

Now the runner around it, written the way a first version is. It says what it is doing, says what it
found, and returns quietly.

In [3]:
def naive_audit(diff):
    """Narration and payload leave through the same channel."""
    print(f"[audit] reviewing {len(diff)} bytes of diff")
    found, spent = review(diff)
    print(f"[audit] {spent.prompt_tokens} tokens in, {spent.completion_tokens} out")
    print(json.dumps(found))
    return 0


buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    code = naive_audit(DIFF)
captured = buffer.getvalue()
print(captured)

try:
    json.loads(captured)         # what the next step in the pipeline does with stdout
    parsed = True
except json.JSONDecodeError as broke:
    parsed = False
    print(f"the pipeline cannot read stdout: {broke}")

print(f"exit code the naive runner returned: {code}")
assert parsed and code != 0, "stdout did not parse, and the runner still returned 0"

[audit] reviewing 454 bytes of diff
[audit] 195 tokens in, 175 out
{"verdict": "block", "findings": [{"rule": "missing-idempotency", "severity": "high", "evidence": "The `refund` function no longer accepts or uses an `idempotency_key`."}, {"rule": "logged-secret", "severity": "high", "evidence": "The `PSP_SECRET_KEY` is logged in the `refund` function."}, {"rule": "unbounded-retry", "severity": "medium", "evidence": "The `refund` function retries indefinitely (99 times) on `PspTimeout` without a backoff strategy."}]}

the pipeline cannot read stdout: Expecting value: line 1 column 2 (char 1)
exit code the naive runner returned: 0


AssertionError: stdout did not parse, and the runner still returned 0

## The diagnosis

The audit was right. It found the removed idempotency key, the credential written into a log line,
and the retry loop with no ceiling. None of it reached anybody.

**The channels are mixed.** `stdout` is a typed channel, and its type is the report. Every friendly
line printed into it is part of the payload, so the parse fails on the first character.

**The exit code was a habit, not a decision.** The runner returned 0 because nothing threw.

Three of the seven rows in the mechanics table are about which channel carries what. This runner
used one channel for all of it.

## The fix

The fix is a file with that contract written into it. Two pieces are worth building first, starting
with the one that keeps a credential out of anything you post.

In [4]:
import re

SECRET_SHAPES = [re.compile(r"sk-[a-z0-9\-]{2,10}-[A-Za-z0-9\-_]{20,}"),
                 re.compile(r"gh[pousr]_[A-Za-z0-9]{30,}")]


def redact(text):
    """Nothing leaves this process with a credential still in it."""
    for shape in SECRET_SHAPES:
        text = shape.sub("[redacted]", text)
    return text


planted = "psp call failed with " + "sk-live-" + "z" * 34
print(f"secret shaped strings found: {sum(bool(s.search(planted)) for s in SECRET_SHAPES)}")
print(f"what gets posted instead   : {redact(planted)}")

secret shaped strings found: 1
what gets posted instead   : psp call failed with [redacted]


The second piece is the validator, which is code that rejects a value that is the right shape but
the wrong answer. Both live in `audit.py` beside this notebook, with the argument parsing and the
exit code mapping. Run it the way the pipeline will.

In [5]:
def run_audit(diff_file, seconds=60):
    """Run the entrypoint exactly as the workflow does. No keyboard, bounded."""
    return subprocess.run(
        [sys.executable, "audit.py", "--diff", diff_file,
         "--schema", "audit-schema.json", "--max-seconds", "60"],
        stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=seconds)

Same diff, same model, same schema. The difference is which channel carries what.

In [6]:
blocked = run_audit("sample-diff.txt")
report = json.loads(blocked.stdout)

print(f"before: stdout did not parse, and the runner returned {code}")
print(f"after : stdout parsed, exit {blocked.returncode}, verdict {report['verdict']!r}")
print(f"stderr: {blocked.stderr.strip()}")
print(f"findings: {[f['rule'] for f in report['findings']]}")

before: stdout did not parse, and the runner returned 0
after : stdout parsed, exit 2, verdict 'block'
stderr: [audit] reviewed in 0.01s, 3 findings, verdict block
findings: ['missing-idempotency', 'logged-secret', 'unbounded-retry']


A change with nothing wrong in it has to come back clean, or the audit blocks every merge and gets
switched off within a week.

In [7]:
passed = run_audit("clean-diff.txt")
verdict = json.loads(passed.stdout)["verdict"]

print(f"clean diff : exit {passed.returncode}, verdict {verdict!r}")
print(f"blocked one: exit {blocked.returncode}")

clean diff : exit 0, verdict 'pass'
blocked one: exit 2


The workflow that calls it is `example-workflow.yml` in this folder. It is not in
`.github/workflows/` here on purpose, because this repository is the teaching material, not the
thing under review. Copy it into the repository you want audited.

Three lines carry the lesson.

```yaml
env:
  OPENROUTER_API_KEY: ${{ secrets.OPENROUTER_API_KEY }}   # one step only
run: |
  uv run python audit.py --diff pr.diff --schema audit-schema.json \
    --max-seconds 120 < /dev/null > report.json 2> audit.log
  echo "code=$?" >> "$GITHUB_OUTPUT"
```

The key is scoped to the one step that needs it and never appears on the command line. `stdin` comes
from `/dev/null`, so nothing can wait for a person. The exit code is captured, and 1 and 2 are
handled in separate steps.

## The gate

One check, covering what the whole vault is about. A pipeline must tell three outcomes apart, and
nothing it posts may carry a credential.

In [8]:
def test_the_pipeline_can_tell_three_outcomes_apart():
    finding = run_audit("sample-diff.txt")
    clean = run_audit("clean-diff.txt")
    broken = run_audit("no-such-file.diff")

    codes = (clean.returncode, broken.returncode, finding.returncode)
    assert codes == (0, 1, 2), f"expected (0, 1, 2) for clean, broken, finding and got {codes}"
    assert not any(s.search(finding.stdout) for s in SECRET_SHAPES), "the report carries a key"
    return codes


print("gate holds, exit codes:", test_the_pipeline_can_tell_three_outcomes_apart())

gate holds, exit codes: (0, 1, 2)


Make `audit.py` return 0 when the audit itself breaks and this test fails on the middle number.
That regression turns a broken audit into a silent approval.

### Enterprise exploration

- Every pull request now waits on a model call. What is the added latency at your merge rate, and
  when does the audit become the bottleneck?
- The audit blocks a merge at midnight and the provider is down. Who can override it, and what does
  that override leave behind for an auditor?
- Findings are advice, not facts. What is your false positive rate, and what happens the day a team
  stops trusting the block?
- The key sits in a repository secret store. What is the blast radius if someone can add a workflow,
  and what would you change?

### Key takeaways

- An audit that reports and does not block is a comment, not a control.
- `stdout` carries the payload and nothing else. Narration belongs on `stderr`.
- Give the audit failing its own exit code, or a broken audit reads as a clean review.
- The key comes from the secret store, is scoped to one step, and never touches argv or the report.